# Feature Engineering for Enterprise Demand Forecasting

## Objective

This notebook transforms the master dataset into a machine-learning-ready dataset by creating predictive features from historical sales, calendar information, and pricing data.

The engineered features help the forecasting model capture:

- Demand persistence
- Weekly and monthly seasonality
- Price sensitivity
- Holiday effects
- Inventory patterns

Output:
- features.parquet

In [11]:
import polars as pl

In [12]:
data = pl.read_parquet(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\data\processed\master_dataset.parquet"
)

print(data.shape)
data.head()
print(data.estimated_size("mb"))

(59181090, 35)
13212.694334030151


In [16]:

data = data.sort(["store_id", "item_id", "date"])

In [17]:
data = data.with_columns([
    pl.col("sales").shift(1).over(["store_id","item_id"]).alias("lag_1"),
    pl.col("sales").shift(7).over(["store_id","item_id"]).alias("lag_7"),
    pl.col("sales").shift(28).over(["store_id","item_id"]).alias("lag_28")
])

In [18]:
data.select([
    "store_id",
    "item_id",
    "date",
    "sales",
    "lag_1",
    "lag_7",
    "lag_28"
]).head(10)

store_id,item_id,date,sales,lag_1,lag_7,lag_28
str,str,str,i64,i64,i64,i64
"""CA_1""","""FOODS_1_001""","""2011-01-29""",3,null,null,null
"""CA_1""","""FOODS_1_001""","""2011-01-30""",0,3,null,null
"""CA_1""","""FOODS_1_001""","""2011-01-31""",0,0,null,null
"""CA_1""","""FOODS_1_001""","""2011-02-01""",1,0,null,null
"""CA_1""","""FOODS_1_001""","""2011-02-02""",4,1,null,null
"""CA_1""","""FOODS_1_001""","""2011-02-03""",2,4,null,null
"""CA_1""","""FOODS_1_001""","""2011-02-04""",0,2,null,null
"""CA_1""","""FOODS_1_001""","""2011-02-05""",2,0,3,null
"""CA_1""","""FOODS_1_001""","""2011-02-06""",0,2,0,null


In [19]:
data = data.with_columns([
    pl.col("sales")
      .shift(1)
      .rolling_mean(window_size=7)
      .over(["store_id","item_id"])
      .alias("rolling_mean_7"),

    pl.col("sales")
      .shift(1)
      .rolling_mean(window_size=28)
      .over(["store_id","item_id"])
      .alias("rolling_mean_28")
])

In [20]:
data = data.with_columns([
    pl.when(pl.col("weekday").is_in(["Saturday","Sunday"]))
      .then(1)
      .otherwise(0)
      .alias("is_weekend"),

    pl.when(pl.col("event_name_1").is_not_null())
      .then(1)
      .otherwise(0)
      .alias("is_event")
])

In [21]:
data = data.with_columns([
    pl.col("sell_price")
      .shift(1)
      .over(["store_id","item_id"])
      .alias("prev_price")
])

data = data.with_columns([
    (
        (pl.col("sell_price") - pl.col("prev_price"))
        / pl.col("prev_price")
    ).alias("price_change_pct")
])

In [22]:
data = data.fill_null(0)

In [23]:
data.write_parquet(
    r"C:\Users\Nikesh\Enterprise-Demand-Forecasting\data\processed\features.parquet"
)

print("✅ Feature engineering completed successfully!")

✅ Feature engineering completed successfully!
